# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all record sets, fields, and columns by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing each by `@id`.

In [ ]:
# List record sets
print('Available record sets and their fields:')
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- Record Set @id: {rs.id}, Name: {rs.name}")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {field.name}, Data Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare record set IDs (by @id)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Extract data into DataFrames, keyed by record set @id
dataframes = {}
for rs_id in record_set_ids:
    # It's safest to load as list into DataFrame, may be empty if no records present
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show DataFrame info for first non-empty record set
chosen_rs_id = next((rs_id for rs_id in record_set_ids if not dataframes[rs_id].empty), None)
if chosen_rs_id is None:
    print("No records found in any record set.")
else:
    print(f"Loaded DataFrame columns for Record Set @id: {chosen_rs_id}")
    print(dataframes[chosen_rs_id].columns.tolist())
    display(dataframes[chosen_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps – such as filtering, normalization, and grouping using field `@id`s. We'll select a numeric field if available.

In [ ]:
import numpy as np

if chosen_rs_id is not None:
    df = dataframes[chosen_rs_id]
    # Find a numeric field (float or integer) by field @id
    chosen_rs = next(rs for rs in dataset.metadata.record_sets if rs.id == chosen_rs_id)
    numeric_field_ids = [field.id for field in chosen_rs.fields if field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number'] and field.id in df.columns]

    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        # Apply filtering (e.g. threshold at mean, if data allows)
        if np.issubdtype(df[numeric_field_id].dtype, np.number):
            threshold = df[numeric_field_id].mean()
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
            # Try grouping by another (categorical/text) field
            group_fields = [field.id for field in chosen_rs.fields if field.data_type=="schema:Text" and field.id in filtered_df.columns]
            if group_fields:
                group_field = group_fields[0]
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
                print(f"Grouped mean of {numeric_field_id} by {group_field}:")
                display(grouped_df.head())
            else:
                print("No suitable group (categorical) field found for grouping.")
        else:
            print(f"Field {numeric_field_id} is not numeric in DataFrame.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No record set with records for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations will reference field columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id is not None and 'numeric_field_id' in locals():
    df = dataframes[chosen_rs_id]
    # Simple histogram
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field available, box plot
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR² dataset using its Croissant schema, explored available record sets and fields, and demonstrated dynamic extraction, filtering, normalization, and visualization. All data manipulation referenced Croissant `@id`s for full reproducibility and clarity. Proceed with domain-specific analysis as appropriate for your work!